In [12]:
!uv pip install langgraph langchain-upstage langchain==1.1.3 rich langchain-text-splitters langchain_classic qdrant_client langchain_qdrant langchain_tavily langchain_community

Using Python 3.13.7 environment at: /home/ejsong/Documents/Projects/.venv
⠙ pydantic==2.12.5                                                              

Resolved 81 packages in 116ms                                        
Installed 7 packages in 42ms0.4.1                           
 + dataclasses-json==0.6.7
 + httpx-sse==0.4.3
 + langchain-community==0.4.1
 + marshmallow==3.26.2
 + mypy-extensions==1.1.0
 + pydantic-settings==2.12.0
 + typing-inspect==0.9.0


In [22]:
import os
from dotenv import load_dotenv

# 1. .env 파일의 내용을 환경 변수로 로드합니다.
load_dotenv()

# 2. 설정할 키 목록
keys = [
    'LANGSMITH_API_KEY', 'UPSTAGE_API_KEY', 'TAVILY_API_KEY'
]

for key in keys:
    # os.getenv는 해당 키가 없으면 None을 반환합니다.
    if not os.getenv(key):
        print(f"경고: {key}가 .env 파일이나 시스템 환경 변수에 설정되어 있지 않습니다.")

# 3. 고정값 설정 (이미 로드된 환경 변수가 있다면 그대로 사용하거나 직접 할당)
os.environ["LANGSMITH_TRACING_V2"] = 'true'
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"] = 'agentic-workflow'

# 확인 출력 (실제 운영 시에는 API Key 출력 금지!)
# print(f"프로젝트명: {os.environ.get('LANGSMITH_PROJECT')}")

In [23]:
from rich import print as rprint
from langchain.chat_models import init_chat_model

MODEL = "solar-pro2"
llm = init_chat_model(
    model=MODEL,
    temperature=0.0
)

In [31]:
!pip install chromadb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 169.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 114.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 170.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 184.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 172.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 24.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42/42 [chromadb]chromadb]jsonschema]ry-api]protos]


In [ ]:
import os
import pandas as pd
import json
from rich import print as rprint

# 1. 라이브러리 교체 (OpenAI 관련 제거, Upstage 및 Chroma 추가)
from langchain_upstage import UpstageEmbeddings, ChatUpstage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_community.vectorstores import Chroma

# [필수] 업스테이지 API 키 설정 (본인의 키를 입력하거나 환경변수에 설정하세요)
# os.environ["UPSTAGE_API_KEY"] = "YOUR_UPSTAGE_API_KEY"

# 1번 셀의 llm 설정을 그대로 사용하되, 혹시 모르니 체크 로직을 넣습니다.
from langchain.chat_models import init_chat_model
MODEL = "solar-pro2"
llm = init_chat_model(model=MODEL, temperature=0.0)

# 1. 지식 베이스 구축 함수
def build_vector_db(file_path):
    rprint(f"[bold blue]데이터 로드 시도 중:[/bold blue] {file_path}")

    # --- CSV 로딩 로직 (기존 유지) ---
    encodings = ['utf-8-sig', 'utf-8', 'cp949', 'euc-kr']
    df = None
    for enc in encodings:
        try:
            df = pd.read_csv(file_path, encoding=enc)
            rprint(f"[green]성공: encoding={enc} 로 파일을 읽었습니다.[/green]")
            break
        except:
            continue

    if df is None:
        raise ValueError("파일을 읽을 수 없습니다. 경로와 인코딩을 확인하세요.")

    # 컬럼 전처리
    df.columns = df.columns.str.strip()
    df['생활가이드'] = df['생활가이드'].fillna("")
    df['식이요법/생활가이드'] = df['식이요법/생활가이드'].fillna("")
    
    # 검색용 텍스트 생성
    df['combined_text'] = df.apply(lambda row: 
        f"질환명: {row['병명']}\n[생활가이드]\n{row['생활가이드']}\n[식이요법]\n{row['식이요법/생활가이드']}", axis=1)
    
    # --- 핵심 수정 부분: UpstageEmbeddings 사용 ---
    try:
        # OpenAIEmbeddings()를 사용하면 안 됩니다. 반드시 UpstageEmbeddings를 사용하세요.
        embeddings = UpstageEmbeddings(model="solar-embedding-1-large")
        
        # FAISS 대신 Chroma 사용
        vector_db = Chroma.from_texts(
            texts=df['combined_text'].tolist(),
            embedding=embeddings,
            collection_name="medical_info"
        )
        rprint("[bold green]Chroma 벡터 DB 구축 완료 (Upstage 임베딩 사용)[/bold green]")
        return vector_db
    except Exception as e:
        rprint(f"[bold red]벡터 DB 생성 오류:[/bold red] {e}")
        raise e


# 2. 메인 에이전트 클래스
class MedicalGuideAgent:
    def __init__(self, vector_db):
        self.llm = llm
        # Chroma 리트리버 설정
        self.retriever = vector_db.as_retriever(search_kwargs={"k": 1})
        self.parser = JsonOutputParser()

    def run(self, dialogue_text):
        # [단계 1] 대화 분석 및 정보 추출
        analysis_prompt = ChatPromptTemplate.from_template("""
        당신은 의료 상담 보조 AI입니다. 다음 환자-의사 대화에서 정보를 추출하세요.
        대화내용: {dialogue}
        
        출력 형식(JSON):
        {{"disease": "질병명", "guide_in_text": "추출된 가이드 내용(없으면 None)"}}
        """)
        
        try:
            chain = analysis_prompt | self.llm | self.parser
            extracted = chain.invoke({"dialogue": dialogue_text})
        except Exception as e:
            return f"LLM 분석 오류: {e}"
        
        disease = extracted.get('disease', 'None')
        guide_in_text = extracted.get('guide_in_text', 'None')

        # [단계 2] 로직 처리
        if guide_in_text != 'None' and len(str(guide_in_text)) > 5:
            return f"★ 대화 내용 기반 가이드:\n{guide_in_text}"
        
        if disease != 'None':
            # Chroma DB 검색
            docs = self.retriever.invoke(disease)
            if docs:
                return f"★ [{disease}] 공식 가이드:\n{docs[0].page_content}"
        
        return "정보를 찾을 수 없습니다. 의사에게 직접 문의하세요."


# 3. 실행부
if __name__ == "__main__":
    file_path = "../data/medical_2.csv" # 본인의 파일명 확인
    
    try:
        # DB 구축
        medical_db = build_vector_db(file_path)
        
        # 에이전트 실행
        agent = MedicalGuideAgent(medical_db)
        
        test_dialogue = "환자: 목이 아파요. 의사: 검사 결과 후두암입니다. 절대 금연하세요."
        result = agent.run(test_dialogue)
        
        rprint("\n[bold yellow]결과:[/bold yellow]")
        print(result)
        
    except Exception as e:
        rprint(f"[bold red]최종 오류 발생:[/bold red] {e}")


데이터 로드 시도 중: medical_2.csv

성공: encoding=cp949 로 파일을 읽었습니다.

Chroma 벡터 DB 구축 완료 (Upstage 임베딩 사용)

결과:

★ 대화 내용 기반 가이드:
절대 금연하세요


In [ ]:
# 3. 실행부
if __name__ == "__main__":
    file_path = "../data/medical_2.csv" # 본인의 파일명 확인
    
    try:
        # DB 구축
        medical_db = build_vector_db(file_path)
        
        # 에이전트 실행
        agent = MedicalGuideAgent(medical_db)
        
        test_dialogue = "환자: 목이 아파요. 의사: 검사 결과 후두암입니다."
        result = agent.run(test_dialogue)
        
        rprint("\n[bold yellow]결과:[/bold yellow]")
        print(result)
        
    except Exception as e:
        rprint(f"[bold red]최종 오류 발생:[/bold red] {e}")

데이터 로드 시도 중: medical_2.csv

성공: encoding=cp949 로 파일을 읽었습니다.

Chroma 벡터 DB 구축 완료 (Upstage 임베딩 사용)

결과:

★ [후두암] 공식 가이드:
질환명: 후두암
[larynx cancer]
[생활가이드]

[식이요법]
흡연은 물론이고 음주, 특히 흡연과 심한 음주를 함께 하는 것은 피하는 것이 좋다. 심한 음주는 단독으로도 후두암 발생에 영향을 미치는 것으로 알려져 있으므로, 음주를 하는 경우에는 음주량을 줄이는 것이 후두암을 예방하는데 도움을 준다. 또한 채소, 과일, 곡물을 많이 섭취하고 비타민 A, C, E 등을 적당량 섭취하는 것 역시 후두암 예방에 효과가 있는 방법이다.


In [35]:
# 에이전트 실행
agent = MedicalGuideAgent(medical_db)
        
test_dialogue = "환자: 목이 아파요. 의사: 간 초음파 검사입니다."
result = agent.run(test_dialogue)
        
rprint("\n[bold yellow]결과:[/bold yellow]")
print(result)

결과:

★ 대화 내용 기반 가이드:
간 초음파 검사


In [ ]:
# # 에러가 발생할 수 있는 주요 원인들을 표기하며 코드에 주석 추가 및 포인트별 위험 설명

# import pandas as pd
# import json
# import os
# from langchain_community.vectorstores import FAISS

# # 다음 임포트에서 에러가 나는 대표적인 원인:
# # 1. openai, langchain_openai, langchain_core 등이 설치 안 됨
# # 2. openai, langchain 등 라이브러리 버전 불일치 (런타임에서 ImportError)
# # 3. OPENAI_API_KEY 등 필수 환경 변수가 없으면 embeddings 생성 안 됨
# # 4. medical.csv 파일이 없거나 잘못된 경로일 때 FileNotFoundError
# # 5. medical.csv가 필수 컬럼, 예: '병명', '생활가이드', '식이요법/생활가이드' 중 하나라도 없을 시 KeyError
# # 6. 프롬프트 결과가 JSON 포맷이 아니거나 key 누락 시 get에서 오류
# from langchain_openai import OpenAIEmbeddings, ChatOpenAI
# from langchain_core.prompts import ChatPromptTemplate
# from langchain_core.output_parsers import JsonOutputParser

# # 1. 지식 베이스 구축 함수 (CSV 파싱 방어 설계)
# def build_vector_db(file_path):
#     print(f"데이터 로드 시도 중: {file_path}")

#     # 한국어 CSV 특성을 고려한 다중 인코딩 + 구분자 + 엔진 조합 시도
#     # - 에러(tokenizing)는 보통 '인코딩'이 아니라 '구분자/따옴표/줄바꿈' 등 CSV 포맷 문제에서 발생
#     encodings = ['utf-8-sig', 'utf-8', 'cp949', 'euc-kr']
#     seps = [',', '\t', ';', '|']

#     df = None
#     last_error = None

#     for enc in encodings:
#         # 1) 흔한 구분자들을 C 엔진으로 빠르게 시도
#         for sep in seps:
#             try:
#                 df = pd.read_csv(file_path, encoding=enc, sep=sep)
#                 print(f"성공: encoding={enc}, sep={repr(sep)} 로 파일을 읽었습니다.")
#                 last_error = None
#                 break
#             except UnicodeDecodeError as e:
#                 last_error = e
#                 df = None
#                 break  # 인코딩 문제면 sep 바꿔봐도 의미가 없어서 다음 encoding으로
#             except FileNotFoundError as e:
#                 raise FileNotFoundError(f"파일 경로 오류: {e}")
#             except Exception as e:
#                 last_error = e
#                 df = None
#                 continue

#         if df is not None:
#             break

#         # 2) 마지막 수단: python 엔진 + 자동 구분자 추정 (느리지만 깨진 CSV에 더 관대)
#         try:
#             df = pd.read_csv(file_path, encoding=enc, sep=None, engine='python')
#             print(f"성공: encoding={enc}, engine='python', sep=None(자동추정) 로 파일을 읽었습니다.")
#             last_error = None
#             break
#         except UnicodeDecodeError as e:
#             last_error = e
#             df = None
#             continue
#         except FileNotFoundError as e:
#             raise FileNotFoundError(f"파일 경로 오류: {e}")
#         except Exception as e:
#             last_error = e
#             df = None
#             continue

#     if df is None:
#         raise ValueError(
#             "medical.csv를 CSV로 파싱하지 못했습니다. "
#             "(인코딩 문제라기보다 구분자/따옴표/깨진 줄바꿈 등 포맷 문제일 가능성이 큽니다)\n"
#             f"마지막 에러: {last_error}"
#         )

#     # 컬럼명 전처리 (공백 제거 및 필수 컬럼 확인)
#     df.columns = df.columns.str.strip()
#     required_columns = ['병명', '생활가이드', '식이요법/생활가이드']
#     for col in required_columns:
#         if col not in df.columns:
#             raise KeyError(f"필수 컬럼 '{col}'이(가) 데이터셋에 존재하지 않습니다.")

#     # 데이터 정제: 결측치 제거 및 검색용 텍스트 생성
#     df['생활가이드'] = df['생활가이드'].fillna("")
#     df['식이요법/생활가이드'] = df['식이요법/생활가이드'].fillna("")
    
#     # [병명]을 기준으로 검색이 잘 되도록 포맷팅
#     df['combined_text'] = df.apply(lambda row: 
#         f"질환명: {row['병명']}\n[생활가이드]\n{row['생활가이드']}\n[식이요법]\n{row['식이요법/생활가이드']}", axis=1)
    
#     # 벡터 DB 생성 (FAISS 사용)
#     try:
#         embeddings = OpenAIEmbeddings()  # 여기에 OPENAI_API_KEY가 없으면 에러 발생
#         vector_db = FAISS.from_texts(df['combined_text'].tolist(), embeddings)
#     except Exception as e:
#         print(f"FAISS/임베딩 생성 오류: {e}")
#         raise e

#     print("벡터 데이터베이스 구축이 완료되었습니다.")
#     return vector_db


# # 2. 메인 에이전트 클래스
# class MedicalGuideAgent:
#     def __init__(self, vector_db):
#         # 'llm' 오브젝트가 글로벌 범위에 없거나 미정의하면 NameError
#         try:
#             self.llm = llm
#         except NameError:
#             raise Exception("llm 오브젝트가 글로벌에 정의되어 있지 않습니다. (예: init_chat_model 통한 초기화 필요)")
#         self.retriever = vector_db.as_retriever(search_kwargs={"k": 1})
#         self.parser = JsonOutputParser()

#     def run(self, dialogue_text):
#         # [단계 1] 대화 분석 및 정보 추출 프롬프트
#         analysis_prompt = ChatPromptTemplate.from_template("""
#         당신은 의료 상담 보조 AI입니다. 다음 환자-의사 대화에서 정보를 추출하세요.
        
#         대화내용: {dialogue}
        
#         다음 지침을 엄격히 따르세요:
#         1. 'disease': 대화에서 언급된 최종 진단명이나 의심되는 병명을 추출하세요. (없으면 "None")
#         2. 'guide_in_text': 의사가 직접 언급한 생활 습관, 운동, 식단 등 가이드 내용이 있다면 요약하세요. (없으면 "None")
        
#         출력 형식(JSON):
#         {{"disease": "질병명", "guide_in_text": "추출된 가이드 내용"}}
#         """)
        
#         # 분석 실행
#         try:
#             chain = analysis_prompt | self.llm | self.parser
#             extracted = chain.invoke({"dialogue": dialogue_text})
#         except Exception as e:
#             print(f"프롬프트 체인 실행 에러: {e}")
#             raise e
        
#         # extracted가 None이거나 dict가 아니면 이후 get에서 AttributeError
#         disease = extracted.get('disease', 'None') if isinstance(extracted, dict) else 'None'
#         guide_in_text = extracted.get('guide_in_text', 'None') if isinstance(extracted, dict) else 'None'

#         # [단계 2] 로직 분기 처리

#         # Case 1: 대화 속에 이미 가이드가 있는 경우 (최우선순위)
#         if guide_in_text != 'None' and len(guide_in_text) > 5:
#             return f"★ 대화 내용 기반 맞춤 가이드:\n{guide_in_text}"
        
#         # Case 2: 대화 속에 가이드가 없어서 RAG 검색을 하는 경우
#         if disease != 'None':
#             # 벡터 DB에서 질병명으로 검색
#             try:
#                 docs = self.retriever.invoke(disease)
#             except Exception as e:
#                 print(f"벡터 DB 질의 오류: {e}")
#                 return f"벡터 DB 질의 중 오류 발생: {e}"
            
#             if docs:
#                 # 검색된 내용이 실제 해당 질병과 관련 있는지 검증 후 출력
#                 # page_content는 AttributeError 가능(객체가 올바르지 않은 경우)
#                 try:
#                     return f"★ 서울대학교병원 의학정보 기반 [{disease}] 가이드:\n{docs[0].page_content}"
#                 except Exception as e:
#                     return f"검색 결과 접근 오류: {e}"
#             else:
#                 # Case 3: DB에도 정보가 없는 경우
#                 return f"전문가 확인 필요: [{disease}]에 대한 상세 생활 가이드 정보가 데이터베이스에 없습니다. 정확한 안내를 위해 담당 의사에게 확인해 주세요."
        
#         # Case 4: 질병 자체를 추출할 수 없는 경우
#         return "진단된 병명이 대화에서 확인되지 않습니다. 의사에게 병명과 생활 수칙을 다시 확인해 주세요."

# # 3. 실제 실행부
# if __name__ == "__main__":
#     # 환경 변수에 OPENAI_API_KEY가 설정되어 있어야 합니다.
#     # os.environ["OPENAI_API_KEY"] = "your-api-key-here"
    
#     file_path = "../data/medical_2.csv"
    
#     try:
#         # 1) 지식 베이스 구축
#         medical_db = build_vector_db(file_path)
        
#         # 2) 에이전트 초기화
#         agent = MedicalGuideAgent(medical_db)
        
#         # 3) 테스트 실행 (대화 내 가이드가 없는 경우 -> RAG 작동)
#         test_dialogue = "환자: 선생님, 검사 결과가 어떻게 나왔나요? 의사: 검사 결과 후두암으로 확인되었습니다. 당분간 무리하지 마세요."
#         result = agent.run(test_dialogue)
        
#         print("\n[에이전트 응답 결과]")
#         print(result)
        
#     except Exception as e:
#         print(f"오류 발생: {e}")

